# A tiny walkthrough of the critical-amplitude controller

This notebook explains the small controller prototype in `critical_search_prototype.py`.
It **does not evolve Engrenage**. It takes the information from completed runs and
uses it to choose a next amplitude.

We will first use a toy scaling law with a known answer. That lets us check the
controller arithmetic without spending time on a collapse simulation. At the end,
we show how a real Engrenage result would be added to the run history.

## The whole flow

1. Choose an amplitude `A` and run the simulation.
2. From that run, get `L` (your central-scalar variation), `dL_dA`, and an outcome.
3. Store those values as one run record.
4. The controller uses the derivatives and outcomes to estimate a scaling slope,
   maintain a subcritical/supercritical bracket, and suggest another `A`.

The controller runs as ordinary Python. JAX is still used inside the simulation
to calculate `dL_dA`; the controller itself does not need to be differentiated.


## 1. What counts as one run record?

One dictionary represents one completed evolution. `A` is the input amplitude,
`L` is the observable from that run, `dL_dA` is its JAX derivative, and `outcome`
is the classification from the evolution.

We use `sub` and `super` as short labels for subcritical and supercritical.
An `ambiguous` run has not yet clearly dispersed or collapsed. It must be extended
or investigated; it must **not** update the bracket.

In [1]:
example_record = {
    "A": 0.25,
    "L": 0.8,
    "dL_dA": 5.0,
    "outcome": "sub",
}
example_record

{'A': 0.25, 'L': 0.8, 'dL_dA': 5.0, 'outcome': 'sub'}

## 2. Make a toy problem whose answer we know

Near the critical amplitude, David models the observable as

`L(A) = a + b log(|A - A_star|)`.

Differentiating gives `dL/dA = b / (A - A_star)`. We will pretend the true
threshold is `A_star = 0.30` and `b = -0.27`. These values are only for this toy
test. The controller below is **not told** either value; it sees only run records.

In [2]:
A_star_for_toy = 0.30
b_for_toy = -0.27

def toy_gradient(A):
    return b_for_toy / (A - A_star_for_toy)

toy_runs = []
for A in (0.20, 0.25, 0.40):
    outcome = "sub" if A < A_star_for_toy else "super"
    toy_runs.append({
        "A": A,
        "L": 1.0,  # L is not needed for the pairwise b estimate
        "dL_dA": toy_gradient(A),
        "outcome": outcome,
    })

for run in toy_runs:
    print(run)

{'A': 0.2, 'L': 1.0, 'dL_dA': 2.6999999999999997, 'outcome': 'sub'}
{'A': 0.25, 'L': 1.0, 'dL_dA': 5.3999999999999995, 'outcome': 'sub'}
{'A': 0.4, 'L': 1.0, 'dL_dA': -2.6999999999999997, 'outcome': 'super'}

## 3. The bracket: one known run on each side

The lower end is the highest amplitude known to be subcritical. The upper end
is the lowest amplitude known to be supercritical. In the toy records, the
highest subcritical amplitude is `0.25` and the lowest supercritical one is
`0.40`, so the unknown threshold is bracketed between them.

The real threshold is not assumed to be the midpoint. The bracket is a safety
interval for proposed updates and a way to report an error bound.

In [3]:
def get_bracket(runs):
    sub = [run["A"] for run in runs if run["outcome"] == "sub"]
    sup = [run["A"] for run in runs if run["outcome"] == "super"]
    if not sub or not sup:
        return None
    lower = max(sub)
    upper = min(sup)
    if lower >= upper:
        raise ValueError("The sub and super runs do not form a valid bracket.")
    return lower, upper

bracket = get_bracket(toy_runs)
print("bracket:", bracket)
print("bracket midpoint:", sum(bracket) / 2)

bracket: (0.25, 0.4)
bracket midpoint: 0.325

## 4. Estimate `b` from the gradients

From `L′(A) = b / (A - A_star)`, take a reciprocal:

`1/L′ = (A - A_star)/b`.

For two runs, subtract these equations. `A_star` cancels, leaving:

`b = (A2 - A1) / (1/L′2 - 1/L′1)`.

For example, use the first two toy runs. The calculation gives `b = -0.27`.
David’s controller uses several useful run pairs and takes a median, so one noisy
pair does not dominate. This works well only when the runs are close enough to the
threshold for the logarithmic scaling law to apply.

In [4]:
A1, g1 = toy_runs[0]["A"], toy_runs[0]["dL_dA"]
A2, g2 = toy_runs[1]["A"], toy_runs[1]["dL_dA"]
b_from_these_two = (A2 - A1) / (1 / g2 - 1 / g1)
print("b from two subcritical runs:", b_from_these_two)

b from two subcritical runs: -0.2700000000000001

## 5. The prototype’s robust `b` estimate

The next cell contains the small implementation from the Python file. It:

- keeps only finite gradients with the expected sign (`L′ > 0` on sub runs and
  `L′ < 0` on super runs for a negative `b`);
- looks at up to six runs with the largest `|L′|` (usually the runs closest to
  threshold);
- forms the usable pairwise estimates and returns their median.

This is deliberately a first prototype. It does not yet have David’s global-fit
veto, echo-count filter, or moving-event-stop correction.

In [5]:
from itertools import combinations
from math import isfinite
from statistics import median

def estimate_b(runs, max_runs=6):
    usable = []
    for run in runs:
        outcome = run["outcome"]
        grad = float(run["dL_dA"])
        amp = float(run["A"])
        right_sign = (outcome == "sub" and grad > 0) or (outcome == "super" and grad < 0)
        if outcome in ("sub", "super") and right_sign and isfinite(grad) and grad != 0 and isfinite(amp):
            usable.append((amp, grad))

    usable = sorted(usable, key=lambda item: abs(item[1]), reverse=True)[:max_runs]
    b_values = []
    for (a1, g1), (a2, g2) in combinations(usable, 2):
        inv_g1, inv_g2 = 1 / g1, 1 / g2
        denominator = inv_g2 - inv_g1
        scale = max(abs(inv_g1), abs(inv_g2), 1e-300)
        if abs(denominator) < 1e-10 * scale or not isfinite(denominator):
            continue
        b = (a2 - a1) / denominator
        if isfinite(b) and b < 0:
            b_values.append(b)

    return median(b_values) if b_values else None

b_hat = estimate_b(toy_runs)
print("estimated b:", b_hat)
print("known toy b:", b_for_toy)

estimated b: -0.27
known toy b: -0.27

## 6. Turn the gradient into a proposed amplitude

The scaling law gives the Newton-like proposal

`A_new = A - b / (dL/dA)`.

Use the last toy run: `A=0.40`, `L′=-2.7`, and `b=-0.27`. The proposal is
`0.40 - (-0.27 / -2.7) = 0.30`. It lies inside the known bracket `(0.25, 0.40)`,
so the controller accepts it.

If there is not enough information to estimate `b`, the prototype makes a
startup probe. If a Newton proposal falls outside the bracket, it falls back to
the bracket midpoint. These are safety/fallback moves, not the desired routine
update.

In [6]:
def propose_next_amplitude(runs):
    if not runs:
        raise ValueError("Add a run before asking for the next amplitude.")
    last = runs[-1]
    if last["outcome"] == "ambiguous":
        raise ValueError("Extend the ambiguous run before choosing another amplitude.")
    if last["outcome"] not in ("sub", "super"):
        raise ValueError("outcome must be 'sub', 'super', or 'ambiguous'.")

    A = float(last["A"])
    grad = float(last["dL_dA"])
    bracket = get_bracket(runs)
    b = estimate_b(runs)
    right_sign = (last["outcome"] == "sub" and grad > 0) or (last["outcome"] == "super" and grad < 0)

    if b is not None and isfinite(grad) and right_sign:
        candidate = A - b / grad
        if bracket is None:
            return candidate, "Newton proposal; no bracket yet", b
        if bracket[0] < candidate < bracket[1]:
            return candidate, "Newton proposal inside bracket", b
        return sum(bracket) / 2, "Newton proposal outside bracket; use midpoint", b

    if bracket is not None:
        return sum(bracket) / 2, "Need a better slope estimate; use midpoint", b
    if last["outcome"] == "sub":
        return 1.15 * A, "Startup probe: increase after subcritical run", b
    return 0.87 * A, "Startup probe: decrease after supercritical run", b

A_next, reason, b_hat = propose_next_amplitude(toy_runs)
print("next amplitude:", A_next)
print("why:", reason)
print("b used:", b_hat)

next amplitude: 0.3
why: Newton proposal inside bracket
b used: -0.27

## 7. Ambiguous runs do not move the bracket

If a run reaches its final time without a clear collapse or dispersion verdict,
the controller refuses to propose a new amplitude from it. First extend or
otherwise resolve that run. This prevents an uncertain outcome from corrupting
the bracket.

In [7]:
ambiguous_history = [
    {"A": 0.30, "L": 1.0, "dL_dA": 10.0, "outcome": "ambiguous"}
]

try:
    propose_next_amplitude(ambiguous_history)
except ValueError as error:
    print(error)

Extend the ambiguous run before choosing another amplitude.

## 8. Convergence: when do we stop?

The notebook describes three stopping checks. `tol_p` is the requested precision
in amplitude, not the time step of the evolution:

1. The known sub/super bracket is narrower than `tol_p`.
2. The proposed step is smaller than `tol_p`, and the estimated threshold has
   changed by less than `tol_p` since the previous estimate.
3. The “cheap certificate”: the step is smaller than `tol_p` and the bracket is
   already narrower than `100 * tol_p`.

A step rejected by the global-fit check cannot be used for checks 2 or 3.

In [8]:
def convergence_status(bracket, last_step, estimate, previous_estimate, tol_p,
                       step_was_vetoed=False):
    if bracket is not None and bracket[1] - bracket[0] < tol_p:
        return True, "bracket is narrower than tol_p"
    if step_was_vetoed:
        return False, "the global-fit check rejected this step"
    if (previous_estimate is not None
            and abs(last_step) < tol_p
            and abs(estimate - previous_estimate) < tol_p):
        return True, "step and estimate change are both below tol_p"
    if (bracket is not None
            and abs(last_step) < tol_p
            and bracket[1] - bracket[0] < 100 * tol_p):
        return True, "cheap certificate"
    return False, "not converged"

tol_p = 1e-10
print(convergence_status(
    bracket=(0.29999999996, 0.30000000001),
    last_step=2e-11,
    estimate=0.30000000000,
    previous_estimate=0.30000000004,
    tol_p=tol_p,
))

(True, 'bracket is narrower than tol_p')

## 9. Where this connects to your Engrenage notebook

You already have a JAX calculation that returns `L` and `dL_dA` from your
`central_scalar_variation(A)` function. After you also obtain a reliable outcome
for that run, a record has this form:

```python
run_record = {
    "A": float(amplitude),
    "L": float(value),
    "dL_dA": float(sensitivity),
    "outcome": "sub",  # or "super"; use "ambiguous" if unclear
}
runs.append(run_record)
```

Then `propose_next_amplitude(runs)` returns `(next_A, reason, b_hat)`.

The notebook you are currently using must supply the actual collapse/dispersal
classification. The older copy available while building this walkthrough did not
include the later event code, so this teaching notebook intentionally does not
guess how your live `solution` signals an event.

## 10. What this prototype does not do yet

- It does not call `evolve` or run a physical simulation.
- It does not decide whether Engrenage’s constraint errors or grid resolution
  are acceptable.
- It does not implement David’s global-fit veto, echo-based slope estimate,
  moving-stop-time correction, or tangent rescaling.
- It assumes `L′` has the expected sign and that the runs used to estimate `b`
  are close enough to the critical scaling regime.

That makes it useful as a transparent first controller, but not yet a validated
critical-amplitude finder. The next integration step is to connect it to the
current event-aware evolution and test `L′` at fixed endpoints before letting
the controller drive physical runs.